In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from scipy import stats
import time
import os

aa_dir = r"C:\Users\user\Downloads\GSE148375_clean"

# Reload gene burden matrix
gene_burden_cpd_pc = pd.read_csv(os.path.join(aa_dir, "gene_burden_matrix_signed_protein_coding_CPD_AA.csv"), index_col=0)
gene_names_cpd = gene_burden_cpd_pc.index.tolist()
smoker_cols = gene_burden_cpd_pc.columns.tolist()

# Reload phenotype
meta_df = pd.read_csv(os.path.join(aa_dir, "checkpoint2b_metadata_relatedness_filtered.csv"))
meta_df["sample_id"] = meta_df["sample_id"].astype(str)
meta_df = meta_df.set_index("sample_id")
pheno_cpd = meta_df.loc[smoker_cols, "cpd"].astype(float)
print("CPD phenotype reloaded:", pheno_cpd.shape)

# Reload confounders
confounders_cpd = np.load(os.path.join(aa_dir, "confounders_X_cpd.npy"))
print("Confounders reloaded:", confounders_cpd.shape)

# DoubleML setup
X_genes_cpd = gene_burden_cpd_pc.T.values
X_standardized_cpd = (X_genes_cpd - X_genes_cpd.mean(axis=0)) / X_genes_cpd.std(axis=0)
Y_cpd = pheno_cpd.values

def doubleml_scan(X_snps, Y, X_conf, n_folds=5, random_state=42):
    n, n_snps = X_snps.shape
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=random_state)
    D_resid = np.zeros_like(X_snps)
    Y_resid = np.zeros(n)
    Xc = np.column_stack([np.ones(n), X_conf])

    for train_idx, test_idx in kf.split(Xc):
        Xc_tr, Xc_te = Xc[train_idx], Xc[test_idx]
        coef_Y = np.linalg.lstsq(Xc_tr, Y[train_idx], rcond=None)[0]
        Y_resid[test_idx] = Y[test_idx] - Xc_te @ coef_Y
        coef_D = np.linalg.lstsq(Xc_tr, X_snps[train_idx], rcond=None)[0]
        D_resid[test_idx] = X_snps[test_idx] - Xc_te @ coef_D

    Yr = Y_resid - Y_resid.mean()
    Dr = D_resid - D_resid.mean(axis=0)
    del D_resid

    sum_DY = (Dr * Yr[:, None]).sum(axis=0)
    sum_DD = (Dr ** 2).sum(axis=0)
    sum_YY = (Yr ** 2).sum()

    theta = sum_DY / sum_DD
    ssr = sum_YY - (sum_DY ** 2) / sum_DD
    sigma2 = ssr / (n - 2)
    se = np.sqrt(sigma2 / sum_DD)
    t_stat = theta / se
    pvals = 2 * (1 - stats.t.cdf(np.abs(t_stat), df=n - 2))
    return theta, pvals

n_repeats = 30
threshold = 0.001
n_genes_cpd = X_standardized_cpd.shape[1]
significant_counts_cpd = np.zeros(n_genes_cpd, dtype=int)

start = time.time()
for rep in range(n_repeats):
    theta_rep, pval_rep = doubleml_scan(X_standardized_cpd, Y_cpd, confounders_cpd, random_state=rep)
    significant_counts_cpd += (pval_rep < threshold).astype(int)
    if (rep + 1) % 5 == 0:
        print(f"Completed {rep+1}/{n_repeats}, elapsed {time.time()-start:.1f}s")

print(f"Total time: {time.time()-start:.1f}s")

stability_fraction_cpd = significant_counts_cpd / n_repeats
results_df_cpd = pd.DataFrame({
    "gene": gene_names_cpd,
    "stability_fraction": stability_fraction_cpd,
    "n_significant_repeats": significant_counts_cpd
}).sort_values("stability_fraction", ascending=False)

print("\nStability distribution:")
for t in [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    print(f"  >= {t:.0%}: {(stability_fraction_cpd >= t).sum()} genes")

results_df_cpd.to_csv(os.path.join(aa_dir, "gene_doubleml_stability_cpd_AA.csv"), index=False)
print("\nSaved.")

CPD phenotype reloaded: (1459,)
Confounders reloaded: (1459, 12)
Completed 5/30, elapsed 6.3s
Completed 10/30, elapsed 12.7s
Completed 15/30, elapsed 19.0s
Completed 20/30, elapsed 25.3s
Completed 25/30, elapsed 31.5s
Completed 30/30, elapsed 37.8s
Total time: 37.8s

Stability distribution:
  >= 50%: 61 genes
  >= 60%: 58 genes
  >= 70%: 57 genes
  >= 80%: 51 genes
  >= 90%: 46 genes
  >= 100%: 33 genes

Saved.


In [3]:
# Dedup check
gene_burden_shortlist_cpd = gene_burden_cpd_pc.loc[results_df_cpd[results_df_cpd["stability_fraction"]==1.0]["gene"]]
corr_cpd = gene_burden_shortlist_cpd.T.corr()
r2_cpd = (corr_cpd ** 2).values.copy()
np.fill_diagonal(r2_cpd, 0)
dup_pairs = np.argwhere(r2_cpd > 0.99)

shortlist_100_cpd = results_df_cpd[results_df_cpd["stability_fraction"]==1.0].copy()
dup_genes_drop = set()
for i, j in dup_pairs:
    if i < j:
        g1, g2 = shortlist_100_cpd["gene"].iloc[i], shortlist_100_cpd["gene"].iloc[j]
        print(f"Duplicate: {g1} <-> {g2}, r²={r2_cpd[i,j]:.4f}")
        dup_genes_drop.add(g2)

shortlist_dedup_cpd = shortlist_100_cpd[~shortlist_100_cpd["gene"].isin(dup_genes_drop)]
print(f"\nAA CPD genes after dedup: {len(shortlist_dedup_cpd)}")

# Try full PC algorithm (no grouping) - should be fast given 34 nodes
import time
import json
from causallearn.search.ConstraintBased.PC import pc
from causallearn.utils.cit import fisherz
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode

genes_list_cpd = shortlist_dedup_cpd.sort_values("stability_fraction", ascending=False)["gene"].tolist()
gene_burden_final_cpd = gene_burden_cpd_pc.loc[genes_list_cpd]
X_genes_pc_cpd = gene_burden_final_cpd.T.values
Y_pc_cpd = pheno_cpd.values.reshape(-1, 1)
X_pc_full_cpd = np.hstack([X_genes_pc_cpd, Y_pc_cpd])
col_names_cpd = gene_burden_final_cpd.index.tolist() + ["smoking_status_placeholder"]
col_names_cpd[-1] = "cpd"  # rename outcome properly

print("AA CPD PC input shape:", X_pc_full_cpd.shape)

n_nodes_cpd = len(col_names_cpd)
outcome_idx_cpd = col_names_cpd.index("cpd")

bk_cpd = BackgroundKnowledge()
nodes_cpd = [GraphNode(name) for name in col_names_cpd]
for i in range(n_nodes_cpd - 1):
    bk_cpd.add_node_to_tier(nodes_cpd[i], 0)
bk_cpd.add_node_to_tier(nodes_cpd[outcome_idx_cpd], 1)

start = time.time()
cg_cpd = pc(
    data=X_pc_full_cpd,
    alpha=0.001,
    indep_test=fisherz,
    stable=True,
    uc_rule=0,
    uc_priority=2,
    background_knowledge=bk_cpd,
    depth=3,
    verbose=False,
    show_progress=True,
    node_names=col_names_cpd
)
elapsed = time.time() - start
print(f"Completed in {elapsed:.1f}s")

adj_cpd = cg_cpd.G.graph
direct_parents_cpd = [col_names_cpd[i] for i in range(n_nodes_cpd) if col_names_cpd[i] != "cpd"
                       and ((adj_cpd[i, outcome_idx_cpd] == -1 and adj_cpd[outcome_idx_cpd, i] == 1)
                            or (adj_cpd[i, outcome_idx_cpd] == -1 and adj_cpd[outcome_idx_cpd, i] == -1))]
print(f"\nDirect parents: {len(direct_parents_cpd)}")
print(direct_parents_cpd)


AA CPD genes after dedup: 33


c:\Users\user\Desktop\ai causal\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AA CPD PC input shape: (1459, 34)


Depth=8, working on node 33: 100%|██████████| 34/34 [00:00<00:00, 1985.94it/s]


Completed in 2.4s

Direct parents: 9
['NCKAP5', 'EXD3', 'TRIM66', 'HYDIN', 'COL18A1', 'ZNF805', 'THNSL2', 'PPP1R12B', 'FAT4']
